# 01 · Multimodal Foundations

> **Source notes:** `MultimodalFoundations.md`

How do images, audio, and video become tensors a neural network can process?

This notebook makes it concrete:
- **Image tensors** — load a JPEG, inspect shape, normalise, visualise channels
- **Audio tensors** — build a mel spectrogram from a synthetic waveform
- **Video tensors** — sample frames from a video and inspect the 4-D tensor shape
- **The modality gap** — show why you cannot simply concatenate image and text tensors

**All local, no API key or GPU needed.** Runtime: < 30 seconds.

## 0 · Setup

```bash
# No Ollama needed for this chapter.
# Packages installed below via pip.
```

In [ ]:
import subprocess, sys
pkgs = ["torch", "torchvision", "Pillow", "matplotlib", "numpy", "torchaudio"]
subprocess.run([sys.executable, "-m", "pip", "install", *pkgs, "-q"], check=True)
print("Packages ready.")

## 1 · Image Tensors

A colour image in PyTorch is a tensor of shape **(C, H, W)** — channels first.

| Dimension | Meaning | Typical size |
|-----------|---------|-------------|
| C         | Colour channels (R, G, B) | 3 |
| H         | Height in pixels | 224 (ViT standard) |
| W         | Width in pixels | 224 |

**Normalisation** converts uint8 $[0, 255]$ to float32 and shifts the distribution so each channel has zero mean and unit variance (ImageNet statistics).

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# ── PixelSmith v0: build a synthetic test image (3-channel gradient) ──────────
H, W = 224, 224
img_np = np.zeros((H, W, 3), dtype=np.uint8)

# Red channel: horizontal gradient
img_np[:, :, 0] = np.linspace(0, 255, W, dtype=np.uint8)[np.newaxis, :]
# Green channel: vertical gradient
img_np[:, :, 1] = np.linspace(0, 255, H, dtype=np.uint8)[:, np.newaxis]
# Blue channel: diagonal gradient
xx, yy = np.meshgrid(np.linspace(0, 255, W), np.linspace(0, 255, H))
img_np[:, :, 2] = ((xx + yy) / 2).astype(np.uint8)

img_pil = Image.fromarray(img_np)

# Step 1: PIL → NumPy (H, W, C)
arr = np.array(img_pil)  # shape: (224, 224, 3)
print(f"PIL → NumPy shape (HWC): {arr.shape}  dtype: {arr.dtype}")

# Step 2: Convert to float32, normalise to [0, 1]
arr_float = arr.astype(np.float32) / 255.0

# Step 3: HWC → CHW  (channels first — PyTorch convention)
arr_chw = np.transpose(arr_float, (2, 0, 1))  # shape: (3, 224, 224)
print(f"NumPy → CHW:             {arr_chw.shape}")

# Step 4: To PyTorch tensor
tensor = torch.from_numpy(arr_chw)            # shape: (3, 224, 224)
print(f"PyTorch tensor shape:    {tensor.shape}  dtype: {tensor.dtype}")

# Step 5: Add batch dimension → (N, C, H, W)
batch = tensor.unsqueeze(0)                   # shape: (1, 3, 224, 224)
print(f"Batched (N=1):           {batch.shape}")

In [ ]:
# Visualise: show each channel separately
fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
channel_names = ["Red channel", "Green channel", "Blue channel"]
cmaps = ["Reds", "Greens", "Blues"]

axes[0].imshow(img_pil)
axes[0].set_title("Original image")
axes[0].axis("off")

for i, (name, cmap) in enumerate(zip(channel_names, cmaps)):
    axes[i + 1].imshow(arr_chw[i], cmap=cmap, vmin=0, vmax=1)
    axes[i + 1].set_title(f"{name}\nshape: (224, 224)")
    axes[i + 1].axis("off")

plt.suptitle("PixelSmith v0 — Image as three independent channel tensors", y=1.02)
plt.tight_layout()
plt.show()

print(f"\nChannel value stats:")
for i, name in enumerate(channel_names):
    ch = arr_chw[i]
    print(f"  {name}: min={ch.min():.3f}  max={ch.max():.3f}  mean={ch.mean():.3f}  std={ch.std():.3f}")

In [ ]:
# ImageNet normalisation — what pretrained models expect
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

# Normalise: (x - mean) / std
tensor_norm = (tensor - IMAGENET_MEAN) / IMAGENET_STD

print("Before ImageNet normalisation:")
for i, name in enumerate(["R", "G", "B"]):
    print(f"  {name}: mean={tensor[i].mean():.3f}  std={tensor[i].std():.3f}")

print("\nAfter ImageNet normalisation (target: mean≈0, std≈1):")
for i, name in enumerate(["R", "G", "B"]):
    print(f"  {name}: mean={tensor_norm[i].mean():.3f}  std={tensor_norm[i].std():.3f}")

print("\nDiffusion model convention [-1, 1]:")
tensor_diffusion = tensor * 2.0 - 1.0
print(f"  min={tensor_diffusion.min():.3f}  max={tensor_diffusion.max():.3f}")

## 2 · Audio Tensors — Mel Spectrograms

Audio models almost never work on raw waveforms. They convert the signal to a **mel spectrogram** — a 2-D frequency-vs-time representation that:
- Compresses the time axis (from 16,000 samples/sec to ~100 frames/sec)
- Uses a perceptual frequency scale (mel) that matches human hearing
- Applies log compression to handle dynamic range

The result looks like an image, which is why ViT-based models (Whisper, AST) can process audio directly.

In [ ]:
import torchaudio
import torchaudio.transforms as T

# ── Synthetic audio: chord of three sine waves ────────────────────────────────
SAMPLE_RATE = 16_000
DURATION_S  = 3.0
t = torch.linspace(0, DURATION_S, int(SAMPLE_RATE * DURATION_S))

# A major chord: A4 (440 Hz), C#5 (554 Hz), E5 (659 Hz)
waveform = (torch.sin(2 * np.pi * 440 * t) +
            torch.sin(2 * np.pi * 554 * t) +
            torch.sin(2 * np.pi * 659 * t)) / 3.0
waveform = waveform.unsqueeze(0)  # shape: (1, T) — mono, batch of 1

print(f"Raw waveform shape: {waveform.shape}  ({waveform.shape[1]:,} samples @ {SAMPLE_RATE} Hz)")
print(f"Duration: {waveform.shape[1] / SAMPLE_RATE:.1f}s")

# Mel spectrogram transform
mel_transform = T.MelSpectrogram(
    sample_rate=SAMPLE_RATE,
    n_fft=512,
    hop_length=160,       # ~10ms frame shift
    n_mels=80,            # 80 mel bins (Whisper uses 80)
)
mel_spec = mel_transform(waveform)  # shape: (1, 80, T')
log_mel  = torch.log(mel_spec + 1e-9)

print(f"\nMel spectrogram shape: {mel_spec.shape}  (batch=1, mels=80, frames={mel_spec.shape[2]})")
print(f"Frames → effective frame rate: {mel_spec.shape[2] / DURATION_S:.1f} frames/sec")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Raw waveform
axes[0].plot(t[:4000].numpy(), waveform[0, :4000].numpy(), lw=0.5)
axes[0].set_title("Raw waveform (first 0.25s)")
axes[0].set_xlabel("Time (s)")
axes[0].set_ylabel("Amplitude")

# Log mel spectrogram
im = axes[1].imshow(log_mel[0].numpy(), aspect="auto", origin="lower",
                    extent=[0, DURATION_S, 0, 80], cmap="viridis")
axes[1].set_title("Log mel spectrogram — shape (80, T')\n"
                  "Treated as a 2-D image by Whisper / AST")
axes[1].set_xlabel("Time (s)")
axes[1].set_ylabel("Mel bin")
plt.colorbar(im, ax=axes[1], label="Log magnitude")

plt.suptitle("Audio → Tensor Pipeline", y=1.02)
plt.tight_layout()
plt.show()

print(f"Log mel stats: min={log_mel.min():.2f}  max={log_mel.max():.2f}  " +
      f"mean={log_mel.mean():.2f}  std={log_mel.std():.2f}")

## 3 · Video Tensors

Video extends images by adding a time dimension. A video tensor has shape **(T, C, H, W)**:

- `T` = number of sampled frames (not all frames — uniform or random sampling)
- `C, H, W` = channels, height, width of each frame

The key challenge: even a 1-second clip at 30fps = 30 frames × 3 × 224 × 224 ≈ 4.5 million values. Models sample a small subset.

In [ ]:
# Simulate video: animated gradient (simple motion)
T_FRAMES = 8
video_frames = []

for t_idx in range(T_FRAMES):
    shift = t_idx * (256 // T_FRAMES)
    frame = np.zeros((224, 224, 3), dtype=np.uint8)
    # Simulate a moving bright region (simple motion)
    frame[:, :, 0] = np.roll(np.linspace(0, 255, 224, dtype=np.uint8), shift)
    frame[:, :, 1] = np.roll(np.linspace(0, 255, 224, dtype=np.uint8), -shift)
    frame[:, :, 2] = 128
    video_frames.append(frame)

# Stack into (T, H, W, C) then convert to (T, C, H, W)
video_np  = np.stack(video_frames)                              # (T, H, W, C)
video_chw = np.transpose(video_np, (0, 3, 1, 2)).astype(np.float32) / 255.0  # (T, C, H, W)
video_t   = torch.from_numpy(video_chw)

print(f"Video tensor shape: {video_t.shape}  (T={T_FRAMES} frames, C=3, H=224, W=224)")
print(f"Memory footprint: {video_t.numel() * 4 / 1024:.1f} KB (float32)")
print(f"\nFor a 1-second 30fps clip at 224×224: "
      f"{30 * 3 * 224 * 224 * 4 / 1024 / 1024:.1f} MB per video")

# Visualise all frames as a strip
fig, axes = plt.subplots(1, T_FRAMES, figsize=(16, 2.5))
for i in range(T_FRAMES):
    frame_img = np.transpose(video_chw[i], (1, 2, 0))  # back to HWC for imshow
    axes[i].imshow(frame_img)
    axes[i].set_title(f"t={i}")
    axes[i].axis("off")

plt.suptitle("Video tensor: 8 frames, each (3, 224, 224). Note: motion between frames.", y=1.05)
plt.tight_layout()
plt.show()

## 4 · The Modality Gap

This cell shows *why* you cannot simply concatenate image and text tensors. The distributions are fundamentally different — they don't live in the same space.

In [ ]:
import torch.nn.functional as F

# Text: token IDs from a simple vocabulary (discrete integers)
VOCAB_SIZE = 32_000
fake_text_tokens = torch.randint(0, VOCAB_SIZE, (1, 16))  # 16 token IDs
print(f"Text token IDs shape: {fake_text_tokens.shape}")
print(f"  Values: {fake_text_tokens[0].tolist()}")
print(f"  Range:  [{fake_text_tokens.min().item()}, {fake_text_tokens.max().item()}]")

# Image: normalised pixel values (continuous floats)
fake_image = torch.rand(1, 3, 8, 8)  # tiny image
print(f"\nImage pixel values shape: {fake_image.shape}")
print(f"  Values (first row of R channel): {fake_image[0, 0, 0].tolist()}")
print(f"  Range:  [{fake_image.min():.3f}, {fake_image.max():.3f}]")

print("\n━━━ The Modality Gap ━━━")
print("Text tokens: discrete integers ∈ [0, 32000]")
print("Image pixels: continuous floats ∈ [0, 1]")
print("\nThey cannot be fed into the same model layer directly.")
print("Solution: each modality needs its own encoder to project into a SHARED embedding space.")
print("\nAfter projection:")

# Both are projected to the same embedding dimension (d=64 here; CLIP uses 512–1024)
D = 64
text_embed  = F.normalize(torch.randn(1, D), dim=-1)  # unit-norm embedding
image_embed = F.normalize(torch.randn(1, D), dim=-1)  # unit-norm embedding

similarity = (text_embed @ image_embed.T).item()
print(f"  Text embedding shape:  {text_embed.shape}  (float, unit norm)")
print(f"  Image embedding shape: {image_embed.shape}  (float, unit norm)")
print(f"  Cosine similarity (untrained): {similarity:.4f}   ← random; CLIP training makes it high for matching pairs")

## 5 · Summary — PixelSmith v0

```
┌─────────────────────────────────────────────────────────────────────────────┐
│ PixelSmith v0 — Input stage                                                  │
│                                                                               │
│  JPEG / PNG  ──load──▶  (H, W, C) uint8                                     │
│                ──CHW──▶  (C, H, W) float32 [0,1]                            │
│                ──norm──▶  (C, H, W) ImageNet-normalised  OR  [-1,1]         │
│                ──batch─▶  (N, C, H, W)                ← ready for Ch.2 ViT │
│                                                                               │
│  Audio .wav ──STFT──▶  (1, F, T') log-mel-spec  ← treated as 2-D image     │
│  Video .mp4 ──sample──▶  (T, C, H, W) per-frame float32                    │
└─────────────────────────────────────────────────────────────────────────────┘
```

**Key takeaways:**
1. All modalities become tensors of floats — but with different shapes and value ranges.
2. Normalisation convention matters: use `-1 to 1` for diffusion, ImageNet stats for ViT/CLIP/ResNet.
3. The modality gap means a shared embedding space is required — which CLIP (Ch.3) provides.
4. Audio models work on mel spectrograms because they look like 2-D images.

**Next:** [VisionTransformers.md](../VisionTransformers/VisionTransformers.md) — how the ViT splits your `(3, H, W)` tensor into patches and applies self-attention.